# V2 push — exam confirmation: SFT v2 seeds 42 + 1234 (ep2)

**Runtime: L4 — MANDATORY.** ~1h, ~8-10 units, checkpointed per seed.

Seed 3407 scored **34.51%** (target 30.40). These two runs produce the 3-seed
mean ± sd — the number that decides whether "a 1.5B beat OctoCoder-16B" is a
claim or an anecdote. Pure peft merging (no unsloth), pinned harness.

In [ ]:
# 1) GPU check
!nvidia-smi -L
import torch
assert torch.cuda.is_bf16_supported(), 'Switch runtime to L4.'
name = torch.cuda.get_device_name(0)
assert 'L4' in name, f'Protocol pins the exam to L4; this is {name}.'
print(name, '- OK')

In [ ]:
# 2) Drive + runs
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
SEEDS = [42, 1234]
def met_path(seed):
    return f'{PHASE8}/metrics_sftv2_ep2_s{seed}.json'
todo = [s for s in SEEDS if not os.path.exists(met_path(s))]
print('to run:', todo)

In [ ]:
%%capture
!pip install -q peft
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 3) Merge (pure peft)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE_ID = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
for seed in todo:
    out = f'/content/final_sftv2_s{seed}'
    if os.path.exists(out):
        continue
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16)
    m = PeftModel.from_pretrained(base, f'{PHASE8}/sft_v2_s{seed}_ep2')
    m = m.merge_and_unload()
    m.save_pretrained(out, safe_serialization=True)
    try:
        tok = AutoTokenizer.from_pretrained(f'{PHASE8}/sft_v2_s{seed}_ep2')
    except Exception:
        tok = AutoTokenizer.from_pretrained(BASE_ID)
    tok.save_pretrained(out)
    del base, m; gc.collect(); torch.cuda.empty_cache()
    print('merged seed', seed)
print('merges done')

In [ ]:
# 4) Harness at the frozen commit
FROZEN_COMMIT = '8fc5bae6479c4fbbb28c3f8b644f6a15b3f3b5bd'
%cd /content
!rm -rf bigcode-evaluation-harness
!git clone -q https://github.com/bigcode-project/bigcode-evaluation-harness.git
%cd /content/bigcode-evaluation-harness
!git checkout -q {FROZEN_COMMIT}
ver = !git rev-parse HEAD
assert ver[0] == FROZEN_COMMIT
!pip install -q -e .
!pip install -q -U transformers accelerate
!pip uninstall -y -q torchao torchaudio torchvision timm
import transformers
print('harness pinned', ver[0][:8], '| transformers', transformers.__version__)

In [ ]:
# 5) THE RUNS
TASK = 'humanevalfixtests-python'
PROMPT = 'instruct'
BATCH = 16
%cd /content/bigcode-evaluation-harness
for seed in todo:
    print('=' * 70)
    print(f'SFT v2 ep2 seed {seed}: 164 x 20')
    gen_path = f'{PHASE8}/gens_sftv2_ep2_s{seed}.json'
    m_path = met_path(seed)
    model_dir = f'/content/final_sftv2_s{seed}'
    !accelerate launch main.py \
      --model {model_dir} --tasks {TASK} --prompt {PROMPT} \
      --do_sample True --temperature 0.2 --top_p 0.95 --n_samples 20 \
      --batch_size {BATCH} --precision bf16 \
      --max_length_generation 2048 --allow_code_execution \
      --save_generations --save_generations_path {gen_path} \
      --metric_output_path {m_path}
    print(open(m_path).read())

In [ ]:
# 6) THE CONFIRMATION TABLE — 3-seed SFT v2 vs the target
import statistics as st
def read_m(p):
    m = json.load(open(p)); k = [x for x in m if x != 'config'][0]
    return m[k].get('pass@1', 0) * 100, m[k].get('pass@10', 0) * 100
paths = {3407: f'{PHASE8}/metrics_sftv2_ep2_s3407.json',
         42: met_path(42), 1234: met_path(1234)}
print(f"{'model':<20} pass@1    pass@10")
print(f"{'base':<20}  17.59%    23.50%")
print(f"{'SFT v1 mean':<20}  24.82%    32.60%")
vals = []
for s, p in paths.items():
    if os.path.exists(p):
        p1, p10 = read_m(p); vals.append((p1, p10))
        print(f"{'SFT v2 ep2 s' + str(s):<20} {p1:6.2f}%   {p10:6.2f}%")
    else:
        print(f'SFT v2 ep2 s{s}: not run')
print(f"{'TARGET OctoCoder-16B':<20}  30.40%")
if len(vals) == 3:
    m1 = st.mean(v[0] for v in vals); s1 = st.stdev([v[0] for v in vals])
    m10 = st.mean(v[1] for v in vals)
    print(f'\nSFT v2 MEAN pass@1: {m1:.2f}% (sd {s1:.2f})   pass@10: {m10:.2f}%')
    print(f'vs target: {m1 - 30.4:+.2f}   |   worst seed: {min(v[0] for v in vals):.2f}%')
    print('\nClaim standard: mean AND every individual seed above 30.4 = clean win;')
    print('mean above but a seed below = report the spread honestly.')
print('\nBring the whole printout back.')

## Bring back to the session
The **confirmation table** — this decides the headline claim. After this:
GRPO v2 (+ random twin) on the v2 checkpoints to chase the 47% pass@10 ceiling.